# LeetCode 3314 — Construct the Minimum Bitwise Array I bằng Python

Link đề: https://leetcode.com/problems/construct-the-minimum-bitwise-array-i/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Array, Math

Skeleton LeetCode:

```python
class Solution:
    def minBitwiseArray(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Biweekly Contest 141); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi số nguyên tố `nums[i]`, tìm số nguyên **nhỏ nhất** `a >= 0` sao cho `a OR (a + 1) == nums[i]`; nếu không có thì ghi `-1`.
2. Nắm các công cụ bit trong Python: `|` (OR), `^` (XOR), `&` (AND), dịch bit `<<` / `>>`, đọc bit thứ `i` bằng `(x >> i) & 1`, và cách xem số nhị phân bằng `bin(x)` / `f"{x:b}"`.
3. Đi từ cách **thử từng `a`** → **tra bảng dựng sẵn** → **duyệt bit đếm số bit 1 cuối** → **công thức `lowbit`** → **thao tác trên chuỗi nhị phân**.
4. Hiểu mấu chốt: `a OR (a + 1)` chỉ làm một việc là **bật bit 0 thấp nhất của `a`** (nên kết quả luôn là số **lẻ**), và để đảo ngược ta **tắt bit 1 nằm ngay dưới chuỗi số 1 cuối** của `x`, tức `ans = x - 2^(t-1)` với `t` là số bit 1 liên tiếp ở cuối.
5. Tránh các lỗi kinh điển: chọn `x - 1` (đúng nhưng **không nhỏ nhất**), tắt nhầm bit (lệch một vị trí), quên xử lý `x = 2` (chẵn), và nhầm `OR` với phép cộng.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` số **nguyên tố**. Hãy dựng mảng `ans` độ dài `n` sao cho với mỗi chỉ số `i`:

- `ans[i] OR (ans[i] + 1) == nums[i]` (phép OR theo bit);
- đồng thời `ans[i]` phải **nhỏ nhất** có thể.

Nếu **không có** giá trị `ans[i]` nào thoả điều kiện thì đặt `ans[i] = -1`.

**Ví dụ 1**

- Input: `nums = [2,3,5,7]`
- Output: `[-1,1,4,3]`
- Giải thích: với `2` không có `a` nào thoả `a OR (a+1) = 2` nên `-1`. Với `3`: `1 OR 2 = 3`. Với `5`: `4 OR 5 = 5`. Với `7`: `3 OR 4 = 7`.

**Ví dụ 2**

- Input: `nums = [11,13,31]`
- Output: `[9,12,15]`

**Ràng buộc:** `1 <= nums.length <= 100`; `2 <= nums[i] <= 1000`; `nums[i]` là số nguyên tố.

Lưu ý: vì "nhỏ nhất" nên với mỗi `nums[i]` đáp án là **duy nhất** (hoặc `-1`). Dù vậy trong phần test ta vẫn viết một hàm **kiểm tra tính hợp lệ** (đúng điều kiện OR và đúng là nhỏ nhất) để độc lập với thuật toán.

In [1]:
import random
from typing import List

examples = [
    ([2, 3, 5, 7], [-1, 1, 4, 3]),
    ([11, 13, 31], [9, 12, 15]),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[2, 3, 5, 7] -> kỳ vọng [-1, 1, 4, 3]
[11, 13, 31] -> kỳ vọng [9, 12, 15]


## 3. Kiến thức nền

### 3.1. Số nhị phân và `bin` / `format`

Máy tính xem số dưới dạng các bit `0`/`1`. Ví dụ `11` là `1011` (nghĩa là `8 + 2 + 1`). Trong Python: `bin(11)` cho `'0b1011'` và `f"{11:b}"` cho `'1011'`; `f"{11:08b}"` thêm số `0` đầu cho đủ 8 chữ số (rất tiện để canh cột).

In [2]:
print(bin(11))                           # 0b1011
print(f"{11:b}", f"{11:08b}")            # 1011 00001011
print(int("1011", 2))                    # 11  (đổi ngược từ chuỗi nhị phân)

0b1011
1011 00001011
11


### 3.2. Phép OR (`|`), AND (`&`), XOR (`^`)

Các phép này làm việc **từng cặp bit cùng vị trí**:

- `a | b` (OR): bit kết quả là `1` nếu **ít nhất một** trong hai bit là 1.
- `a & b` (AND): bit kết quả là `1` nếu **cả hai** bit là 1.
- `a ^ b` (XOR): bit kết quả là `1` nếu hai bit **khác nhau**.

Ví dụ `5 = 101` và `3 = 011`: OR là `111 = 7`, AND là `001 = 1`, XOR là `110 = 6`.

In [3]:
a, b = 5, 3
print(f"{a:03b} | {b:03b} = {a | b:03b} ({a | b})")    # 101 | 011 = 111 (7)
print(f"{a:03b} & {b:03b} = {a & b:03b} ({a & b})")    # 101 & 011 = 001 (1)
print(f"{a:03b} ^ {b:03b} = {a ^ b:03b} ({a ^ b})")    # 101 ^ 011 = 110 (6)

101 | 011 = 111 (7)
101 & 011 = 001 (1)
101 ^ 011 = 110 (6)


### 3.3. Dịch bit `<<`, `>>` và đọc bit thứ `i`

- `1 << i` là số chỉ có **bit thứ `i`** bằng 1 (bằng `2^i`). Bit thấp nhất là bit số `0`.
- `x >> i` dịch `x` sang phải `i` vị trí (bỏ `i` bit thấp).
- `(x >> i) & 1` đọc **bit thứ `i`** của `x`.
- `x ^ (1 << i)` **đảo** bit thứ `i`; nếu bit đó đang là 1 thì kết quả là **tắt** nó (tức trừ đi `2^i`).

In [4]:
x = 11                                   # 1011
for i in range(4):
    print(f"bit {i} của {x} là {(x >> i) & 1}")      # bit 0:1, 1:1, 2:0, 3:1
print(1 << 3, 1 << 0)                    # 8 1
print(f"{x:04b} -> {x ^ (1 << 1):04b} ({x ^ (1 << 1)})")   # tắt bit 1: 1011 -> 1001 (9)

bit 0 của 11 là 1
bit 1 của 11 là 1
bit 2 của 11 là 0
bit 3 của 11 là 1
8 1
1011 -> 1001 (9)


### 3.4. Chẵn / lẻ bằng bit và số nguyên tố

Bit 0 của `x` cho biết `x` **lẻ** (`1`) hay **chẵn** (`0`): `x & 1`. Số nguyên tố duy nhất là số chẵn là `2`; mọi số nguyên tố khác đều **lẻ**. Điều này giúp ta tách riêng trường hợp `2`.

In [5]:
for x in [2, 3, 5, 7, 11, 13]:
    print(x, "lẻ" if x & 1 else "chẵn", f"{x:b}")

2 chẵn 10
3 lẻ 11
5 lẻ 101
7 lẻ 111
11 lẻ 1011
13 lẻ 1101


### 3.5. Giá trị mặc định với `next(..., default)`

`next(iterator, default)` lấy phần tử đầu tiên của một dãy lazy, và nếu dãy **rỗng** thì trả về `default`. Rất hợp cho "tìm số nhỏ nhất thoả điều kiện, không có thì `-1`".

In [6]:
print(next((a for a in range(10) if a * a > 20), -1))   # 5
print(next((a for a in range(10) if a * a > 99), -1))   # -1

5
-1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xem `a OR (a + 1)` làm gì. In bảng cho `a = 0..15`:

In [7]:
print(" a    a+1   a|(a+1)")
for a in range(16):
    print(f"{a:2d} {a:05b} {a + 1:05b} {a | (a + 1):05b} = {a | (a + 1)}")

 a    a+1   a|(a+1)
 0 00000 00001 00001 = 1
 1 00001 00010 00011 = 3
 2 00010 00011 00011 = 3
 3 00011 00100 00111 = 7
 4 00100 00101 00101 = 5
 5 00101 00110 00111 = 7
 6 00110 00111 00111 = 7
 7 00111 01000 01111 = 15
 8 01000 01001 01001 = 9
 9 01001 01010 01011 = 11
10 01010 01011 01011 = 11
11 01011 01100 01111 = 15
12 01100 01101 01101 = 13
13 01101 01110 01111 = 15
14 01110 01111 01111 = 15
15 01111 10000 11111 = 31


Quan sát:

- Kết quả luôn là số **lẻ** (bit 0 luôn là 1). Với `x = 2` (chẵn) ⇒ **không có** `a` ⇒ `-1`.
- Tính tay `x = 11`: thử `a = 0, 1, ..., 10`: `a | (a+1)` lần lượt là `1, 3, 3, 7, 5, 7, 7, 15, 9, 11, 11`. Giá trị `11` xuất hiện **lần đầu** tại `a = 9` (rồi lại tại `a = 10`) ⇒ nhỏ nhất là `9`.
- Chú ý `a = 9 = 1001` và `a = 10 = 1010` đều cho `11`: có thể có **nhiều** `a` thoả, ta lấy **nhỏ nhất**.

In [8]:
x = 11
print([a | (a + 1) for a in range(x)])                    # [1, 3, 3, 7, 5, 7, 7, 15, 9, 11, 11]
print([a for a in range(x) if a | (a + 1) == x])          # [9, 10]  -> nhỏ nhất là 9

[1, 3, 3, 7, 5, 7, 7, 15, 9, 11, 11]
[9, 10]


## 5. Cách 1 — Brute Force / baseline: thử từng `a`

**Ý tưởng.** Với mỗi `x = nums[i]`, thử `a = 0, 1, 2, ...`; số đầu tiên thoả `a | (a + 1) == x` chính là nhỏ nhất. Vì `a | (a + 1) >= a + 1 > a` nên nếu có đáp án thì `a < x`, ta chỉ cần thử `a` trong `range(x)`. Không tìm thấy thì `-1`.

In [9]:
def min_bitwise_brute(nums: List[int]) -> List[int]:
    ans = []
    for x in nums:
        found = -1
        for a in range(x):               # đáp án nếu có thì a < x
            if a | (a + 1) == x:
                found = a                # a đầu tiên chính là nhỏ nhất
                break
        ans.append(found)
    return ans


for nums, expected in examples:
    print(min_bitwise_brute(nums), "(kỳ vọng", expected, ")")

[-1, 1, 4, 3] (kỳ vọng [-1, 1, 4, 3] )
[9, 12, 15] (kỳ vọng [9, 12, 15] )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)` và `M` là giá trị lớn nhất (`<= 1000`). Mỗi phần tử thử tối đa `M` giá trị ⇒ **Time `O(n · M)`**, **Space `O(1)` phụ** (không kể mảng kết quả).

**Nút thắt:** với `n <= 100` và `M <= 1000` ổn (`10^5` thao tác), nhưng nếu `M` lên `10^9` (đúng bài **II**) thì không thể thử từng số. Ta cần một công thức dựa vào cấu trúc bit.

## 6. Ý tưởng thuật toán / bước đột phá

Viết `a` dưới dạng **đuôi toàn số 1**: `a = P 0 1^m` — tức tiền tố `P` (tuỳ ý), một bit `0`, rồi `m` bit `1` ở cuối (`m >= 0`). Khi đó:

- `a + 1 = P 1 0^m` (cộng 1 làm `m` bit 1 cuối thành 0 và bit `0` kia thành 1).
- `a OR (a + 1) = P 1 1^m = P 1^(m+1)`.

Tức phép `a OR (a + 1)` chỉ **bật bit 0 thấp nhất của `a`**. Từ đó, nếu `x` có `t` bit `1` liên tiếp ở cuối (`t >= 1`, `x` lẻ):

- Các `a` thoả đúng là `x` **tắt đi một trong `t` bit 1 cuối** (tắt bit thứ `m` với `m = 0, 1, ..., t - 1`), tức `a = x - 2^m`. Có đúng `t` nghiệm.
- Muốn **nhỏ nhất** ta tắt bit **cao nhất** trong số đó: `m = t - 1`, cho **`a = x - 2^(t-1)`** (= `x ^ (1 << (t - 1))`).
- Nếu `x` chẵn thì `t = 0`, **không có nghiệm** ⇒ `-1`.

Kiểm tra: `11 = 1011` có `t = 2` bit 1 cuối ⇒ nghiệm `11 - 2^0 = 10` và `11 - 2^1 = 9`; nhỏ nhất `9`. Khớp với mục 4.

In [10]:
for x in [11, 13, 31, 3, 7, 5]:
    t = 0
    while (x >> t) & 1:                  # đếm số bit 1 liên tiếp từ bit thấp nhất
        t += 1
    solutions = [x - (1 << m) for m in range(t)]       # tắt từng bit 1 cuối
    print(f"x={x:2d} ({x:05b}) t={t} nghiệm={sorted(solutions)} nhỏ nhất={x - (1 << (t - 1))}")
    assert all(a | (a + 1) == x for a in solutions)

x=11 (01011) t=2 nghiệm=[9, 10] nhỏ nhất=9
x=13 (01101) t=1 nghiệm=[12] nhỏ nhất=12
x=31 (11111) t=5 nghiệm=[15, 23, 27, 29, 30] nhỏ nhất=15
x= 3 (00011) t=2 nghiệm=[1, 2] nhỏ nhất=1
x= 7 (00111) t=3 nghiệm=[3, 5, 6] nhỏ nhất=3
x= 5 (00101) t=1 nghiệm=[4] nhỏ nhất=4


## 7. Các cách tiếp theo

### Cách 2 — Dựng sẵn bảng tra `a | (a + 1) -> a nhỏ nhất`

**Giải thích từng đoạn.**

1. Vì `nums[i] <= 1000` nên mọi `a` cần xét đều `< 1000`. Ta duyệt `a = 0, 1, ..., 1023` **tăng dần** một lần.
2. `table.setdefault(a | (a + 1), a)` chỉ lưu `a` **lần đầu** gặp giá trị `a | (a + 1)`; vì duyệt tăng dần nên đó chính là `a` nhỏ nhất.
3. Với mỗi `x`, tra `table.get(x, -1)`.

Đây là cách "đổi thời gian lấy bộ nhớ", rất dễ nhớ và khó sai.

In [11]:
def build_table(limit: int = 1024) -> dict:
    table = {}
    for a in range(limit):
        table.setdefault(a | (a + 1), a)  # lần đầu gặp = a nhỏ nhất
    return table


TABLE = build_table()
print({x: TABLE[x] for x in [3, 5, 7, 11, 13, 31]})     # {3:1, 5:4, 7:3, 11:9, 13:12, 31:15}
print(TABLE.get(2, -1))                                  # -1


def min_bitwise_table(nums: List[int]) -> List[int]:
    return [TABLE.get(x, -1) for x in nums]


for nums, expected in examples:
    print(min_bitwise_table(nums), "(kỳ vọng", expected, ")")

{3: 1, 5: 4, 7: 3, 11: 9, 13: 12, 31: 15}
-1
[-1, 1, 4, 3] (kỳ vọng [-1, 1, 4, 3] )
[9, 12, 15] (kỳ vọng [9, 12, 15] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionTable:
    def minBitwiseArray(self, nums: List[int]) -> List[int]:
        table = {}
        for a in range(1024):
            table.setdefault(a | (a + 1), a)
        return [table.get(x, -1) for x in nums]


print(SolutionTable().minBitwiseArray([2, 3, 5, 7]))     # [-1, 1, 4, 3]

[-1, 1, 4, 3]


**Độ phức tạp Cách 2:** dựng bảng `O(M)` một lần, mỗi truy vấn `O(1)` ⇒ **Time `O(M + n)`**, **Space `O(M)`**. (Chỉ hợp khi `M` nhỏ như bài này.)

### Cách 3 — Duyệt bit: đếm số bit 1 ở cuối rồi tắt bit `t - 1`

**Giải thích từng đoạn.** Đây là cách dùng đúng phát hiện ở mục 6:

1. Đếm `t`: `while (x >> t) & 1: t += 1`.
2. Nếu `t == 0` (`x` chẵn) ⇒ `-1`.
3. Ngược lại `ans = x ^ (1 << (t - 1))` (hoặc `x - (1 << (t - 1))`).

Vòng đếm chạy tối đa khoảng 10 lần với `x <= 1000`.

In [13]:
def min_bitwise_scan(nums: List[int]) -> List[int]:
    ans = []
    for x in nums:
        t = 0
        while (x >> t) & 1:              # đếm số bit 1 liên tiếp ở cuối
            t += 1
        if t == 0:                       # x chẵn -> vô nghiệm
            ans.append(-1)
        else:
            ans.append(x ^ (1 << (t - 1)))    # tắt bit 1 cao nhất trong cụm cuối
    return ans


for nums, expected in examples:
    print(min_bitwise_scan(nums), "(kỳ vọng", expected, ")")

[-1, 1, 4, 3] (kỳ vọng [-1, 1, 4, 3] )
[9, 12, 15] (kỳ vọng [9, 12, 15] )


In [14]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionScan:
    def minBitwiseArray(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            if x & 1 == 0:
                ans.append(-1)
                continue
            t = 0
            while (x >> t) & 1:
                t += 1
            ans.append(x ^ (1 << (t - 1)))
        return ans


print(SolutionScan().minBitwiseArray([11, 13, 31]))      # [9, 12, 15]

[9, 12, 15]


**Độ phức tạp Cách 3:** mỗi phần tử tốn `O(log M)` ⇒ **Time `O(n log M)`**, **Space `O(1)`** phụ.

### Cách 4 — Công thức `lowbit` không vòng lặp

**Giải thích.** Với `x` lẻ có `t` bit 1 cuối, `x + 1` có `t` bit 0 cuối rồi một bit 1 ở vị trí `t`. Do đó **bit 1 thấp nhất của `x + 1`** là `2^t`. Trick quen thuộc: `lowbit(y) = y & -y` (lấy bit 1 thấp nhất của `y`). Vậy `2^(t-1) = lowbit(x + 1) >> 1` và:

> `ans = x ^ (lowbit(x + 1) >> 1)`

Hãy xem với `x = 11 = 1011`: `x + 1 = 1100`, `lowbit = 0100 = 4`, `4 >> 1 = 2`, `11 ^ 2 = 9`.

In [15]:
x = 11
y = x + 1
print(f"x + 1 = {y:05b}, lowbit(x + 1) = {y & -y} ({y & -y:05b})")   # 4
print("ans =", x ^ ((y & -y) >> 1))                              # 9


def min_bitwise_lowbit(nums: List[int]) -> List[int]:
    ans = []
    for x in nums:
        if x & 1 == 0:
            ans.append(-1)
        else:
            y = x + 1
            ans.append(x ^ ((y & -y) >> 1))
    return ans


for nums, expected in examples:
    print(min_bitwise_lowbit(nums), "(kỳ vọng", expected, ")")

x + 1 = 01100, lowbit(x + 1) = 4 (00100)
ans = 9
[-1, 1, 4, 3] (kỳ vọng [-1, 1, 4, 3] )
[9, 12, 15] (kỳ vọng [9, 12, 15] )


In [16]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành Solution khi nộp)
class SolutionLowbit:
    def minBitwiseArray(self, nums: List[int]) -> List[int]:
        return [-1 if x & 1 == 0 else x ^ (((x + 1) & -(x + 1)) >> 1) for x in nums]


print(SolutionLowbit().minBitwiseArray([2, 3, 5, 7]))    # [-1, 1, 4, 3]

[-1, 1, 4, 3]


**Độ phức tạp Cách 4:** **Time `O(n)`** (mỗi phần tử `O(1)` phép bit), **Space `O(1)`** phụ.

### Cách 5 — Thao tác trên chuỗi nhị phân

**Giải thích.** Dành cho ai thấy bit khó hình dung: đổi `x` thành chuỗi `'...01111'`. Số `t` bit 1 cuối = `len(s) - len(s.rstrip("1"))` (`rstrip("1")` bỏ các chữ `1` ở **cuối** chuỗi). Chuỗi đáp án = phần đầu (bỏ `t` chữ cuối) + `"0"` + `"1" * (t - 1)`; rồi đổi lại số bằng `int(..., 2)`.

Ví dụ `x = 11`: `s = "1011"`, `t = 2`, phần đầu `"10"`, đáp án `"10" + "0" + "1"` = `"1001"` = 9.

In [17]:
def min_bitwise_string(nums: List[int]) -> List[int]:
    ans = []
    for x in nums:
        s = bin(x)[2:]                   # bỏ tiền tố '0b'
        t = len(s) - len(s.rstrip("1"))  # số chữ '1' ở cuối chuỗi
        if t == 0:
            ans.append(-1)
        else:
            head = s[:len(s) - t]        # phần đứng trước cụm 1 cuối (có thể rỗng)
            ans.append(int(head + "0" + "1" * (t - 1), 2))
    return ans


for nums, expected in examples:
    print(min_bitwise_string(nums), "(kỳ vọng", expected, ")")
print(min_bitwise_string([3, 7]))        # [1, 3]  (head rỗng vẫn đúng)

[-1, 1, 4, 3] (kỳ vọng [-1, 1, 4, 3] )
[9, 12, 15] (kỳ vọng [9, 12, 15] )
[1, 3]


**Độ phức tạp Cách 5:** **Time `O(n log M)`** (tạo chuỗi dài `~log M`), **Space `O(log M)`** cho chuỗi tạm.

## 8. TRACE từng bước

**Bảng trace Cách 3** với `nums = [11, 13, 2]`:

| x | nhị phân | bit 0 | bit 1 | bit 2 | `t` | tắt bit | ans |
|---|---|---|---|---|---|---|---|
| 11 | `1011` | 1 | 1 | 0 | 2 | bit 1 | `1001` = 9 |
| 13 | `1101` | 1 | 0 | — | 1 | bit 0 | `1100` = 12 |
| 2 | `0010` | 0 | — | — | 0 | không có | -1 |

Cột `t` dừng ngay ở bit `0` đầu tiên gặp được; cột "tắt bit" là `t - 1`.

In [18]:
def trace_scan(nums: List[int]) -> List[int]:
    ans = []
    for x in nums:
        print(f"x = {x} = {x:b}")
        t = 0
        while True:
            bit = (x >> t) & 1
            print(f"   t={t}: bit {t} = {bit}")
            if not bit:
                break                    # gặp bit 0 -> dừng đếm
            t += 1
        if t == 0:
            print("   t = 0 -> x chẵn, không có nghiệm: -1")
            ans.append(-1)
        else:
            res = x ^ (1 << (t - 1))
            print(f"   t = {t}: tắt bit {t - 1}: {x:b} ^ {1 << (t - 1):b} = {res:b} = {res}")
            print(f"   kiểm tra: {res} | {res + 1} = {res | (res + 1)}")
            ans.append(res)
    return ans


assert trace_scan([11, 13, 2]) == [9, 12, -1]

x = 11 = 1011
   t=0: bit 0 = 1
   t=1: bit 1 = 1
   t=2: bit 2 = 0
   t = 2: tắt bit 1: 1011 ^ 10 = 1001 = 9
   kiểm tra: 9 | 10 = 11
x = 13 = 1101
   t=0: bit 0 = 1
   t=1: bit 1 = 0
   t = 1: tắt bit 0: 1101 ^ 1 = 1100 = 12
   kiểm tra: 12 | 13 = 13
x = 2 = 10
   t=0: bit 0 = 0
   t = 0 -> x chẵn, không có nghiệm: -1


In [19]:
def trace_brute(x: int) -> int:
    print(f"x = {x}: thử a = 0..{x - 1}")
    for a in range(x):
        v = a | (a + 1)
        mark = "  <-- khớp, dừng" if v == x else ""
        print(f"   a={a:2d}  a|(a+1) = {a:05b} | {a + 1:05b} = {v:2d}{mark}")
        if v == x:
            return a
    print("   không có a nào -> -1")
    return -1


assert trace_brute(11) == 9
assert trace_brute(2) == -1

x = 11: thử a = 0..10
   a= 0  a|(a+1) = 00000 | 00001 =  1
   a= 1  a|(a+1) = 00001 | 00010 =  3
   a= 2  a|(a+1) = 00010 | 00011 =  3
   a= 3  a|(a+1) = 00011 | 00100 =  7
   a= 4  a|(a+1) = 00100 | 00101 =  5
   a= 5  a|(a+1) = 00101 | 00110 =  7
   a= 6  a|(a+1) = 00110 | 00111 =  7
   a= 7  a|(a+1) = 00111 | 01000 = 15
   a= 8  a|(a+1) = 01000 | 01001 =  9
   a= 9  a|(a+1) = 01001 | 01010 = 11  <-- khớp, dừng
x = 2: thử a = 0..1
   a= 0  a|(a+1) = 00000 | 00001 =  1
   a= 1  a|(a+1) = 00001 | 00010 =  3
   không có a nào -> -1


## 9. Code nộp LeetCode

Ta nộp **Cách 3** (duyệt bit) vì đủ nhanh, dễ giải thích; bản `lowbit` ngắn hơn nhưng khó đọc hơn với người mới. Nếu muốn gọn hơn, hãy thay phần thân bằng `lowbit` (Cách 4).

In [20]:
class Solution:
    def minBitwiseArray(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            if x & 1 == 0:                   # chẵn (chỉ có thể là 2): không có a
                ans.append(-1)
                continue
            t = 0
            while (x >> t) & 1:              # đếm số bit 1 liên tiếp ở cuối
                t += 1
            ans.append(x ^ (1 << (t - 1)))   # tắt bit 1 cao nhất của cụm cuối
        return ans


sol = Solution()
print(sol.minBitwiseArray([2, 3, 5, 7]))     # [-1, 1, 4, 3]
print(sol.minBitwiseArray([11, 13, 31]))     # [9, 12, 15]

[-1, 1, 4, 3]
[9, 12, 15]


## 10. Vì sao đúng

Đặt `x` lẻ, và `t` là số bit 1 liên tiếp ở cuối `x` (`t >= 1`).

**(a) Mọi nghiệm có dạng `x - 2^m` với `0 <= m < t`.** Nếu `a OR (a + 1) = x`, viết `a = P 0 1^m` như mục 6 thì `x = P 1^(m+1)`; bit 0..`m` của `x` đều là 1 nên `m + 1 <= t`, tức `m <= t - 1`. Ngược lại `a = P 0 1^m` chính là `x` với bit thứ `m` bị tắt (vì bit `m` của `x` là 1), suy ra `a = x - 2^m`.

**(b) Mọi `m` trong `[0, t - 1]` đều cho nghiệm thật.** Tắt bit `m` (đang là 1, và các bit thấp hơn đều là 1) ⇒ `a = P 0 1^m`, rồi `a OR (a + 1) = P 1^(m+1) = x`.

**(c) Nhỏ nhất:** `x - 2^m` nhỏ nhất khi `2^m` lớn nhất, tức `m = t - 1`.

**(d) Vô nghiệm khi `x` chẵn:** `a OR (a + 1)` luôn có bit 0 bằng 1 (một trong `a`, `a + 1` là số lẻ).

Vì mọi `nums[i] >= 2` nguyên tố nên chỉ `2` là chẵn ⇒ đúng điều kiện trong code (`x & 1 == 0` chỉ xảy ra với `x = 2`). Trong test ta cũng kiểm chứng định lý "có đúng `t` nghiệm" bằng vét cạn.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `M` = giá trị lớn nhất trong `nums` (`<= 1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn từng `a` | `O(n · M)` | `O(1)` phụ |
| 2. Bảng tra dựng sẵn | `O(M + n)` | `O(M)` |
| 3. Duyệt bit đếm `t` | `O(n log M)` | `O(1)` phụ |
| 4. Công thức `lowbit` | `O(n)` | `O(1)` phụ |
| 5. Chuỗi nhị phân | `O(n log M)` | `O(log M)` |

Với `n <= 100`, `M <= 1000` mọi cách đều tức thì; Cách 3/4 mới scale được khi `M` rất lớn.

## 12. Bộ test

Vì đề cho "nhỏ nhất", đáp án là duy nhất; tuy vậy ta viết **hàm kiểm tra tính hợp lệ độc lập** `is_valid_answer(nums, ans)`: với mỗi phần tử, nếu `ans[i] = -1` thì **không** có `a` nào thoả; nếu không thì `ans[i] | (ans[i] + 1) == nums[i]` **và** không có `a` nhỏ hơn cũng thoả. Hàm này không dùng bất kỳ công thức bit nào ở trên.

In [21]:
def is_valid_answer(nums, ans):
    if len(ans) != len(nums):
        return False
    for x, a in zip(nums, ans):
        all_solutions = [b for b in range(0, x + 2) if b | (b + 1) == x]
        if a == -1:
            if all_solutions:            # có nghiệm mà vẫn trả -1 là sai
                return False
        else:
            if a < 0 or a | (a + 1) != x:
                return False
            if all_solutions and a != min(all_solutions):    # không phải nhỏ nhất
                return False
    return True


impls = {
    "brute": min_bitwise_brute,
    "table": min_bitwise_table,
    "scan": min_bitwise_scan,
    "lowbit": min_bitwise_lowbit,
    "string": min_bitwise_string,
    "leetcode": Solution().minBitwiseArray,
}


def check_all(nums, label, expected=None):
    for name, fn in impls.items():
        actual = fn(list(nums))
        assert is_valid_answer(nums, actual), (label, name, nums, actual)
        if expected is not None:
            assert actual == expected, (label, name, nums, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, f"ví dụ {idx}", expected)

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Gồm: chỉ có `2`; số nguyên tố nhỏ nhất lẻ `3`; số nguyên tố dạng `2^k - 1` (Mersenne: `3, 7, 31, 127`) có toàn bit 1; số nguyên tố dạng `2^k + 1` (`5, 17, 257`) chỉ có một bit 1 cuối; số lớn nhất `997`; mảng độ dài 1 và 100.

In [22]:
check_all([2], "chỉ có 2", [-1])
check_all([3], "số nguyên tố lẻ nhỏ nhất", [1])
check_all([3, 7, 31, 127], "Mersenne")
check_all([5, 17, 257], "dạng 2^k + 1", [4, 16, 256])
check_all([997], "lớn nhất ràng buộc")
check_all([2, 2, 2], "toàn số 2", [-1, -1, -1])
check_all([127], "toàn bit 1 (127 = 1111111)", [63])
check_all([2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31], "những số nguyên tố đầu tiên")
check_all([997] * 100, "độ dài 100")

PASS chỉ có 2
PASS số nguyên tố lẻ nhỏ nhất
PASS Mersenne
PASS dạng 2^k + 1
PASS lớn nhất ràng buộc
PASS toàn số 2
PASS toàn bit 1 (127 = 1111111)
PASS những số nguyên tố đầu tiên
PASS độ dài 100


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi: (a) chọn `x - 1` (hợp lệ nhưng **không nhỏ nhất**); (b) tắt nhầm bit `t` thay vì `t - 1`; (c) quên trường hợp `x = 2`. Ta dùng `is_valid_answer` để thấy chúng bị bắt.

In [23]:
def wrong_x_minus_1(nums):
    return [x - 1 for x in nums]         # SAI: hợp lệ nhưng không nhỏ nhất (và sai với 2)


def wrong_off_by_one(nums):
    ans = []
    for x in nums:
        if x & 1 == 0:
            ans.append(-1)
            continue
        t = 0
        while (x >> t) & 1:
            t += 1
        ans.append(x ^ (1 << t))         # SAI: bit t là bit 0, XOR sẽ BẬT nó lên
    return ans


def wrong_no_even_case(nums):
    ans = []
    for x in nums:
        t = 0
        while (x >> t) & 1:
            t += 1
        ans.append(x ^ (1 << max(t - 1, 0)))     # SAI: x chẵn vẫn bị gán một giá trị
    return ans


for fn in (wrong_x_minus_1, wrong_off_by_one, wrong_no_even_case):
    out = fn([2, 3, 11])
    print(fn.__name__, out, "hợp lệ?", is_valid_answer([2, 3, 11], out))
    assert not is_valid_answer([2, 3, 11], out)

wrong_x_minus_1 [1, 2, 10] hợp lệ? False
wrong_off_by_one [-1, 7, 15] hợp lệ? False
wrong_no_even_case [3, 1, 9] hợp lệ? False


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc, và kiểm chứng định lý

(1) **Mọi** số nguyên tố trong `[2, 1000]` (168 số) so với tất cả cách. (2) Mảng ngẫu nhiên độ dài `1..100` gồm số nguyên tố trong `[2, 1000]` (sinh bằng sàng Eratosthenes). (3) **Định lý** ở mục 10: với mọi số lẻ `x`, số nghiệm `a` đúng bằng `t` (số bit 1 cuối).

In [24]:
# sàng Eratosthenes: danh sách số nguyên tố <= 1000
sieve = [True] * 1001
sieve[0] = sieve[1] = False
for i in range(2, 32):
    if sieve[i]:
        for j in range(i * i, 1001, i):
            sieve[j] = False
primes = [i for i in range(2, 1001) if sieve[i]]
print(len(primes), primes[:8], primes[-3:])      # 168 ... [983, 991, 997]
assert len(primes) == 168

# (1) mọi số nguyên tố, từng số một và gộp cả mảng
check_all(primes, "tất cả 168 số nguyên tố trong một mảng")
for p in primes:
    for name, fn in impls.items():
        assert is_valid_answer([p], fn([p])), (name, p)
print("PASS từng số nguyên tố một")

# (2) ngẫu nhiên, có seed
rng = random.Random(3314)
for trial in range(400):
    n = rng.randint(1, 100)
    nums = [rng.choice(primes) for _ in range(n)]
    assert 1 <= len(nums) <= 100 and all(2 <= x <= 1000 for x in nums)
    results = [fn(list(nums)) for fn in impls.values()]
    assert is_valid_answer(nums, results[0])
    assert all(r == results[0] for r in results), nums
print("PASS 400 mảng ngẫu nhiên")

# (3) định lý: số nghiệm = số bit 1 liên tiếp ở cuối với mọi số lẻ x (không chỉ số nguyên tố)
for x in range(1, 1025, 2):
    t = 0
    while (x >> t) & 1:
        t += 1
    cnt = sum(1 for a in range(x + 1) if a | (a + 1) == x)
    assert cnt == t, (x, cnt, t)
print("PASS định lý: số nghiệm = t")

168 [2, 3, 5, 7, 11, 13, 17, 19] [983, 991, 997]
PASS tất cả 168 số nguyên tố trong một mảng


PASS từng số nguyên tố một


PASS 400 mảng ngẫu nhiên
PASS định lý: số nghiệm = t


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chọn `x - 1` vì "nó cũng thoả" nhưng không nhỏ nhất

In [25]:
def sai_x_tru_1(x):
    return x - 1                         # SAI: với x lẻ thì (x-1) | x == x nhưng chưa chắc nhỏ nhất


x = 11
print("sai:", sai_x_tru_1(x), "| kiểm tra:", sai_x_tru_1(x) | (sai_x_tru_1(x) + 1), "| đúng:", Solution().minBitwiseArray([x])[0])

sai: 10 | kiểm tra: 11 | đúng: 9


**Vì sao sai:** `10 | 11 = 11` nên `10` hợp lệ, nhưng `9` cũng hợp lệ và **nhỏ hơn**. Đề đòi nhỏ nhất. `x - 1` chỉ đúng khi `t = 1` (như `5, 13, 29, ...`). **Cách sửa:** tắt bit 1 **cao nhất** trong cụm bit 1 cuối, tức `x - 2^(t-1)`.

### Lỗi 2: Lệch một vị trí bit (dùng `t` thay vì `t - 1`)

In [26]:
def sai_lech_bit(x):
    t = 0
    while (x >> t) & 1:
        t += 1
    return x ^ (1 << t)                  # SAI: bit t đang là 0 -> XOR làm nó thành 1


x = 11
a = sai_lech_bit(x)
print("sai:", a, f"({a:b})", "| a | (a+1) =", a | (a + 1), "(phải bằng", x, ")")

sai: 15 (1111) | a | (a+1) = 31 (phải bằng 11 )


**Vì sao sai:** sau vòng đếm, `t` là chỉ số của **bit 0 đầu tiên**; bit `t - 1` mới là bit 1 cao nhất của cụm cuối. XOR với `1 << t` **bật** bit 0 thành 1, tạo số lớn hơn `x` (`11 → 15`) và `15 | 16 = 31 ≠ 11`. **Cách sửa:** dùng `1 << (t - 1)`; luôn thử bằng một ví dụ cụ thể như `11` hoặc `13`.

### Lỗi 3: Quên trường hợp `x = 2` (chẵn)

In [27]:
def sai_quen_so_chan(x):
    t = 0
    while (x >> t) & 1:
        t += 1
    return x ^ (1 << (t - 1))            # với x = 2: t = 0 nên 1 << -1 -> ValueError


try:
    print(sai_quen_so_chan(2))
except ValueError as e:
    print("ValueError:", e)
print("đúng:", Solution().minBitwiseArray([2]))   # [-1]

ValueError: negative shift count
đúng: [-1]


**Vì sao sai:** với `x` chẵn, bit 0 đã là 0 nên `t = 0`; `1 << (t - 1)` là dịch bit **âm** ⇒ lỗi `negative shift count`. Về toán học thì cũng không có nghiệm. **Cách sửa:** kiểm tra `x & 1 == 0` (hoặc `t == 0`) trước và trả về `-1`.

### Lỗi 4: Nhầm OR với phép cộng

In [28]:
def sai_dung_phep_cong(x):
    return (x - 1) // 2                  # SAI: giải a + (a+1) = x thay vì a | (a+1) = x


for x in [3, 11]:
    a = sai_dung_phep_cong(x)
    print(f"x={x}: a={a}, a+(a+1)={a + a + 1}, a|(a+1)={a | (a + 1)}  (đúng phải ra {x})")

x=3: a=1, a+(a+1)=3, a|(a+1)=3  (đúng phải ra 3)
x=11: a=5, a+(a+1)=11, a|(a+1)=7  (đúng phải ra 11)


**Vì sao sai:** `a + (a + 1) = 2a + 1` cho kết quả khác hẳn `a | (a + 1)`: phép OR **không cộng dồn** mà chỉ gộp bit (bit đã là 1 thì giữ nguyên). Với `x = 3` hai phép tình cờ khớp (`a = 1`) nhưng với `x = 11` thì lệch. **Cách sửa:** luôn nghĩ theo bit (bảng ở mục 4), và test bằng `x` đủ lớn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy `a OR (a + 1)` / `a AND (a + 1)` / `a XOR (a - 1)`: nghĩ ngay tới **chuỗi bit 1 (hoặc 0) ở cuối**. `a + 1` biến cụm `0 1…1` thành `1 0…0`; `a - 1` làm ngược lại.
- Bit 1 thấp nhất cần tìm: `y & -y` (lowbit). Số bit 0 cuối: `(y & -y).bit_length() - 1`.
- Bài "nhỏ nhất thoả `f(a) = x`" mà `x` lớn ⇒ đừng vét cạn; liệt kê **tất cả** nghiệm trên vài ví dụ nhỏ, nhìn quy luật (như mục 4 và 6), rồi **chứng minh**.
- Với ràng buộc nhỏ (`<= 1000`), vét cạn / bảng tra là lựa chọn an toàn để **đối chiếu** công thức của bạn.
- Chỉ số bit: bit 0 là bit **thấp nhất** (bên phải). Dễ nhầm "bit thứ `t`" với "bit thứ `t - 1`": hãy luôn test với `11` và `13`.

## 15. Biến thể của bài

1. **Construct the Minimum Bitwise Array II (3315, Medium):** cùng đề nhưng `nums[i]` lên tới `10^9`, `n` lên tới `10^5` ⇒ vét cạn bị TLE; công thức bit của bài này là đáp án. (Ràng buộc chi tiết: đối chiếu lại đề gốc.)
2. **Đáp án lớn nhất** thay vì nhỏ nhất: tắt bit `0` (`m = 0`) ⇒ `x - 1` với `x` lẻ.
3. **Đếm số nghiệm** `a` của `a OR (a + 1) = x`: bằng `t`, số bit 1 liên tiếp ở cuối.
4. Không giới hạn nguyên tố: công thức đúng với **mọi** `x` (chẵn ⇒ `-1`).
5. Bài liên quan: **Number of 1 Bits (191)**, **Power of Two (231)**, **Single Number (136)** — đều chơi với `x & (x - 1)`, `x & -x`.

In [29]:
def max_bitwise_array(nums):
    # nghiệm lớn nhất = tắt bit 0 (m = 0)
    return [-1 if x & 1 == 0 else x - 1 for x in nums]


def count_solutions(x):
    # số nghiệm a của a | (a+1) == x
    t = 0
    while (x >> t) & 1:
        t += 1
    return t


print(max_bitwise_array([2, 3, 5, 7, 11]))               # [-1, 2, 4, 6, 10]
print([count_solutions(x) for x in [2, 3, 5, 7, 11, 31]])    # [0, 2, 1, 3, 2, 5]
for x in range(1, 300):
    brute = [a for a in range(x + 1) if a | (a + 1) == x]
    assert count_solutions(x) == len(brute)
    if brute:
        assert max(brute) == max_bitwise_array([x])[0]
# công thức đúng cả với số không nguyên tố, ví dụ 9, 15, 21
print(Solution().minBitwiseArray([9, 15, 21]))           # [8, 7, 20]
assert Solution().minBitwiseArray([9, 15, 21]) == [8, 7, 20]

[-1, 2, 4, 6, 10]
[0, 2, 1, 3, 2, 5]
[8, 7, 20]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử `a = 0..x-1` | `O(n·M)` | `O(1)` | để hiểu đề, đối chiếu |
| 2. Bảng tra | dựng `a|(a+1) → a` nhỏ nhất một lần | `O(M+n)` | `O(M)` | `M` nhỏ, nhiều truy vấn |
| 3. Duyệt bit | đếm `t`, tắt bit `t-1` | `O(n log M)` | `O(1)` | **nên nộp**, dễ giải thích |
| 4. `lowbit` | `x ^ (lowbit(x+1) >> 1)` | `O(n)` | `O(1)` | khi muốn code ngắn nhất |
| 5. Chuỗi nhị phân | cắt/ghép chuỗi `'0'`/`'1'` | `O(n log M)` | `O(log M)` | dễ hình dung, không dùng bit |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 | 3`, `6 ^ 3`, `12 >> 2`, `1 << 4`, `13 & 1`.

**Bài 2 (tính tay).** Tìm đáp án của `nums = [19, 23, 29]`: viết nhị phân, đếm `t`, tắt bit rồi kiểm tra `a | (a + 1)`.

**Bài 3 (điền TODO).** Hoàn thành `count_trailing_ones(x)` (đếm số bit 1 liên tiếp ở cuối) và `min_bitwise_todo(nums)`.

In [30]:
def count_trailing_ones(x):
    # TODO 1: t = 0, rồi while (x >> t) & 1: t += 1
    # TODO 2: trả về t
    pass


def min_bitwise_todo(nums):
    # TODO 3: với mỗi x: nếu t == 0 -> -1, ngược lại x ^ (1 << (t - 1))
    pass

# KHÔNG gọi các hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_bitwise_lowbit2(nums)` theo Cách 4 **không dùng vòng `while`**: dùng `y = x + 1` và `y & -y`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_bitwise_array(nums)` trả về nghiệm **lớn nhất** (hoặc `-1`). (b) Viết `all_solutions(x)` trả về **danh sách tăng dần** mọi `a` thoả `a OR (a + 1) = x` bằng công thức `x - 2^m`, rồi so với vét cạn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7`, `5`, `3`, `16`, `1`. Bài 3: vòng `while (x >> t) & 1` dừng ở bit 0 đầu tiên gặp.
- **Mức 2 (vừa):** Bài 2: `19 = 10011` (t = 2) → `17`; `23 = 10111` (t = 3) → `19`; `29 = 11101` (t = 1) → `28`. Bài 4: bit 1 thấp nhất của `x + 1` là `2^t`; dịch phải 1 bit ra `2^(t-1)`.
- **Mức 3 (sâu):** Bài 5(a): nghiệm lớn nhất ứng với `m = 0` ⇒ `x - 1` (khi `x` lẻ). Bài 5(b): `[x - (1 << m) for m in range(t)]`, nhớ **sắp xếp tăng dần**.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert 5 | 3 == 7
assert 6 ^ 3 == 5
assert 12 >> 2 == 3
assert 1 << 4 == 16
assert 13 & 1 == 1

# Bài 2
assert Solution().minBitwiseArray([19, 23, 29]) == [17, 19, 28]
for a, x in [(17, 19), (19, 23), (28, 29)]:
    assert a | (a + 1) == x


# Bài 3
def count_trailing_ones_ans(x):
    t = 0
    while (x >> t) & 1:
        t += 1
    return t


def min_bitwise_todo_ans(nums):
    ans = []
    for x in nums:
        t = count_trailing_ones_ans(x)
        ans.append(-1 if t == 0 else x ^ (1 << (t - 1)))
    return ans


for nums, expected in examples:
    assert min_bitwise_todo_ans(nums) == expected


# Bài 4
def min_bitwise_lowbit2(nums):
    ans = []
    for x in nums:
        y = x + 1
        ans.append(-1 if x & 1 == 0 else x ^ ((y & -y) >> 1))
    return ans


for nums, expected in examples:
    assert min_bitwise_lowbit2(nums) == expected


# Bài 5(a)
def max_bitwise_array_ans(nums):
    return [-1 if x & 1 == 0 else x - 1 for x in nums]


assert max_bitwise_array_ans([2, 3, 5, 7]) == [-1, 2, 4, 6]


# Bài 5(b)
def all_solutions(x):
    t = count_trailing_ones_ans(x)
    return sorted(x - (1 << m) for m in range(t))


for x in range(1, 500):
    assert all_solutions(x) == [a for a in range(x + 1) if a | (a + 1) == x]
assert all_solutions(11) == [9, 10]
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm bit 1 cuối rồi tắt bit `t - 1`)

```python
class Solution:
    def minBitwiseArray(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            if x & 1 == 0:
                ans.append(-1)
                continue
            t = 0
            while (x >> t) & 1:
                t += 1
            ans.append(x ^ (1 << (t - 1)))
        return ans
```

**Checklist trước khi submit**

- [ ] `x` chẵn (chỉ có thể là `2`) đã trả `-1` trước khi dịch bit?
- [ ] Tắt bit `t - 1` (bit 1 **cao nhất** của cụm cuối), không phải bit `t`?
- [ ] Đáp án là số **nhỏ nhất** (không phải `x - 1` tuỳ tiện)?
- [ ] Không sửa mảng `nums` đầu vào, trả về list mới?
- [ ] Đã thử `[2, 3, 5, 7]`, `[11, 13, 31]`, `[2]`, `[3]`, `[997]`?
- [ ] Có kiểm tra lại bằng `a | (a + 1) == x` cho vài giá trị?